# 需求文档与 UI 设计评估

本 notebook 用于读取毕业设计管理平台需求文档，提取关键功能与界面需求，并将当前 UI 设计与需求进行对比。

## 1. 分析需求文档

读取 `.docx` 需求文档并提取关键需求。

In [ ]:
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

docx_path = Path('湖南石化职院2026年毕业设计及实习管理平台服务采购项目采购需求+功能清单.docx')
print('DOCX exists:', docx_path.exists())

def read_docx_text(path: Path) -> str:
    # Read the document.xml from .docx (zip) and extract all w:t text nodes
    with zipfile.ZipFile(path, 'r') as z:
        xml = z.read('word/document.xml')
    ns = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}
    root = ET.fromstring(xml)
    texts = [t.text for t in root.iterfind('.//w:t', ns) if t.text]
    # Join with newlines where paragraph breaks are implied by w:p boundaries
    paras = []
    for p in root.iterfind('.//w:p', ns):
        tlist = [t.text for t in p.iterfind('.//w:t', ns) if t.text]
        if tlist:
            paras.append(''.join(tlist))
    return '\n'.join(paras)

if docx_path.exists():
    raw_text = read_docx_text(docx_path)
    print(raw_text[:1500])
else:
    print('需求文档未找到，请确认文件名和路径。')

## 2. 提取 UI 需求

从需求文档中识别主要界面需求和功能模块，并结构化为可对比的数据。

In [ ]:
def extract_requirements(text: str):
    keywords = ['登录','报表','管理','审批','档案','统计','上传','打卡','计划','教师','学生','企业','资料','导入','导出','上报','审核','分配','权限','待办']
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    reqs = []
    for line in lines:
        if any(k in line for k in keywords):
            reqs.append(line)
    # Deduplicate while preserving order
    seen = set(); uniq = []
    for r in reqs:
        if r not in seen:
            seen.add(r); uniq.append(r)
    return uniq[:200]

if 'raw_text' in globals():
    reqs = extract_requirements(raw_text)
    for i, req in enumerate(reqs[:50], 1):
        print(f'{i}. {req}')
else:
    print('未加载文档内容，无法提取需求。')

## 3. 对照设计稿与需求

描述当前 UI 设计中的主要模块，并判断是否覆盖需求中的关键功能。

In [ ]:
ui_design = {
    '登录页': ['身份选择', '账号密码', '微信扫码', '忘记密码'],
    '侧边栏': ['概览', '毕业设计公示', '任务时间管理', '指导老师管理', '学生档案管理', '质量监控', '数据上传省厅', '实习计划审批', '实习分配管理', '打卡签到', '统计分析', '实习档案', '基础数据管理'],
    '顶部栏': ['系统切换', '搜索', '通知', '帮助'],
    '图表与统计': ['统计卡片', '表格列表', '分页'],
    '表单': ['基础信息输入', '搜索过滤', '审核操作']
}
print('当前 UI 设计模块：')
for key, items in ui_design.items():
    print('\n=== {} ==='.format(key))
    for item in items:
        print('-', item)

## 4. 验证 UI 布局合理性

评估当前页面布局是否满足常见的可用性标准，如导航清晰、信息层次明确、功能入口完整。

In [ ]:
def evaluate_layout():
    checks = [
        ('导航清晰', '侧边栏包含主要功能分类'),
        ('多角色登录', '登录页支持身份选择和密码登录'),
        ('功能覆盖', '毕业设计与岗位实习模块都可进入'),
        ('信息层次', '卡片、列表和顶部栏分区明确'),
        ('可扩展性', '侧边栏和页面模块易于追加新功能')
    ]
    for title, desc in checks:
        print(f'{title}: {desc}')

evaluate_layout()

## 5. 生成改进建议

输出当前设计与需求对比后的优化点。

In [ ]:
suggestions = [
    '确认需求文档中的每个业务流程是否已有对应页面入口，例如实习企业管理、材料审核、成绩归集等。',
    '如果需求强调数据上报省平台，建议在 UI 中单独增加“上报进度”与“上传记录”模块。',
    '检查登录页角色切换后是否同步切换侧边栏权限和页面，避免不同身份进入同一页面。',
    '任务与审批类页面应支持筛选/状态管理，当前概览页可补充“待办事项”卡片。',
    '如果需求有移动端或微信小程序使用场景，当前宽屏布局需要补充自适应方案。',
]
for s in suggestions:
    print('-', s)

## 总结与改进建议

- 当前 UI 覆盖了多数核心模块，但需在以下方面补足：
  - 明确“上报进度”与“上传记录”页面。
  - 增加“待办事项/审批”卡片并支持筛选与状态管理。
  - 登录后根据身份动态调整侧边栏权限。
  - 移动端自适应与微信场景优化。

下方代码将尝试列出未在 UI 设计中显式出现但在需求提取中频繁出现的条目，作为进一步核对的参考。

In [ ]:
# 简单对照：列出提取到但 UI 中未显式出现的需求句子（示例输出，需先运行上方单元）
if 'reqs' in globals():
    ui_text = ' '.join([k + ' ' + ' '.join(v) for k,v in ui_design.items()])
    missing = []
    for r in reqs[:200]:
        # 若需求句中的关键信息在 ui 文本中未出现，则视为可能未覆盖
        tokens = [t for t in r.replace('，',' ').replace('。',' ').split() if len(t)>1]
        if tokens and not any(tok in ui_text for tok in tokens):
            missing.append(r)
    print('可能未覆盖的需求（示例）:')
    for m in missing[:30]:
        print('-', m)
else:
    print('未检测到 reqs 变量——请先运行提取需求的单元。')